# Traffic Speed Visualization
Visualizes road segment average speeds using MapboxGL.

In [201]:
# 1. Install Dependencies (if needed)
# !pip install requests pandas geopandas mapboxgl shapely


In [202]:
# 2. Setup
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from mapboxgl.viz import LinestringViz
from mapboxgl.utils import create_color_stops

load_dotenv()

MAPBOX_TOKEN = os.getenv("MAPBOX_TOKEN", "your_mapbox_token_here")
BASE_URL = "http://localhost:8000"

In [203]:
# 3. One-time bootstrap (if API/data are not ready)
# Run these in a terminal from repo root:
# make up
# make migrate
# make ingest


In [204]:
# 4. Request Aggregated Data
params = {
    "day": "Monday",
    "period": "AM Peak",
    "limit": 1000,
}
response = requests.get(f"{BASE_URL}/aggregates/", params=params)
response.raise_for_status()
geojson_data = response.json()
print(f"Fetched {len(geojson_data)} road segments")


Fetched 1000 road segments


In [205]:
# 5. Visualize in Mapbox
features = [
    {
        "type": "Feature",
        "geometry": f["geometry"],
        "properties": {
            "road_name": f["road_name"],
            "link_id": f["link_id"],
            "average_speed": f["average_speed"],
            "length_m": f["length_m"],
        },
    }
    for f in geojson_data
]

viz = LinestringViz(
    {"type": "FeatureCollection", "features": features},
    access_token=MAPBOX_TOKEN,
    color_property="average_speed",
    color_stops=create_color_stops([10, 20, 30, 40, 50], colors="RdYlGn"),
    center=(-81.6557, 30.3322),
    zoom=11,
    line_width_default=1.5,
    opacity=0.8,
)
viz.show()

In [206]:
# 6. Tabular Summary — 10 slowest road segments
df = pd.DataFrame([
    {
        "link_id": f["link_id"],
        "avg_speed": f["average_speed"],
        "road_name": f["road_name"],
        "length_m": f["length_m"],
    }
    for f in geojson_data
])
df.sort_values("avg_speed").head(10)

,link_id,avg_speed,road_name,length_m
536,1021821186,0.62,Swelo Rd,16.9
541,1021905649,1.24,Regency Court Mall,20.6
795,1033382080,1.24,None,15.5
673,1032710453,1.24,Mt Herman St,120.5
719,1032971696,1.24,None,87.9
979,1049745903,1.55,Bartram Office Park,83.4
79,1006497018,1.55,Irongate Dr,76.5
740,1032980614,1.71,Saints Rd,76.1
744,1032983340,1.86,Kingsbury St,22.2
916,1046257776,1.86,Grasse St,191.5


## Spatial Filter
Query road segments within a custom bounding box. Adjust the coordinates and re-run.

In [207]:
# 7. Spatial Filter — adjust these parameters and re-run
BBOX_DAY    = "Monday"
BBOX_PERIOD = "AM Peak"
BBOX_MIN_LON = -81.70
BBOX_MIN_LAT =  30.20
BBOX_MAX_LON = -81.60
BBOX_MAX_LAT =  30.35

resp = requests.post(
    f"{BASE_URL}/aggregates/spatial_filter/",
    json={
        "day": BBOX_DAY,
        "period": BBOX_PERIOD,
        "bbox": [BBOX_MIN_LON, BBOX_MIN_LAT, BBOX_MAX_LON, BBOX_MAX_LAT],
    },
    params={"limit": 1000},
)
resp.raise_for_status()
bbox_data = resp.json()
print(f"Found {len(bbox_data)} road segments in bbox")

Found 1000 road segments in bbox


In [208]:
# 8. Visualize spatial filter results
import math
import numpy as np

bbox_features = [
    {
        "type": "Feature",
        "geometry": f["geometry"],
        "properties": {
            "road_name": f["road_name"],
            "link_id": f["link_id"],
            "average_speed": f["average_speed"],
            "length_m": f["length_m"],
            "_average_speed": f["average_speed"],  # internal color field, hidden from popup
        },
    }
    for f in bbox_data
]

# Bounding box outline
bbox_features.append({
    "type": "Feature",
    "geometry": {
        "type": "LineString",
        "coordinates": [
            [BBOX_MIN_LON, BBOX_MIN_LAT],
            [BBOX_MAX_LON, BBOX_MIN_LAT],
            [BBOX_MAX_LON, BBOX_MAX_LAT],
            [BBOX_MIN_LON, BBOX_MAX_LAT],
            [BBOX_MIN_LON, BBOX_MIN_LAT],
        ],
    },
    "properties": {"road_name": "Bounding Box", "_average_speed": -1},
})

# Grid lines — cover the visible map area (2x bbox padding on each side)
lon_span = BBOX_MAX_LON - BBOX_MIN_LON
lat_span = BBOX_MAX_LAT - BBOX_MIN_LAT
raw_interval = max(lon_span, lat_span) / 5
magnitude = 10 ** math.floor(math.log10(raw_interval))
grid_interval = round(round(raw_interval / magnitude) * magnitude, 6)

pad = max(lon_span, lat_span)
ext_min_lon = BBOX_MIN_LON - pad
ext_max_lon = BBOX_MAX_LON + pad
ext_min_lat = BBOX_MIN_LAT - pad
ext_max_lat = BBOX_MAX_LAT + pad

grid_lons = []
grid_lats = []

for lon in np.arange(
    math.ceil(ext_min_lon / grid_interval) * grid_interval,
    ext_max_lon,
    grid_interval,
):
    lon = round(lon, 6)
    grid_lons.append(lon)
    bbox_features.append({
        "type": "Feature",
        "geometry": {
            "type": "LineString",
            "coordinates": [[lon, ext_min_lat], [lon, ext_max_lat]],
        },
        "properties": {"road_name": f"{lon:.4f}", "_average_speed": -2},
    })

for lat in np.arange(
    math.ceil(ext_min_lat / grid_interval) * grid_interval,
    ext_max_lat,
    grid_interval,
):
    lat = round(lat, 6)
    grid_lats.append(lat)
    bbox_features.append({
        "type": "Feature",
        "geometry": {
            "type": "LineString",
            "coordinates": [[ext_min_lon, lat], [ext_max_lon, lat]],
        },
        "properties": {"road_name": f"{lat:.4f}", "_average_speed": -2},
    })

center_lon = (BBOX_MIN_LON + BBOX_MAX_LON) / 2
center_lat = (BBOX_MIN_LAT + BBOX_MAX_LAT) / 2
zoom = round(math.log2(360 / max(lon_span, lat_span)) + 0.5, 1)

print(f"Bounding box:  SW ({BBOX_MIN_LAT:.4f}, {BBOX_MIN_LON:.4f})  ->  NE ({BBOX_MAX_LAT:.4f}, {BBOX_MAX_LON:.4f})")
print(f"Grid interval: {grid_interval} deg   center ({center_lat:.4f}, {center_lon:.4f})   zoom {zoom}")
print(f"Longitudes: {[f'{v:.4f}' for v in grid_lons]}")
print(f"Latitudes:  {[f'{v:.4f}' for v in grid_lats]}")

color_stops = [
    [-2, "#aaaaaa"],   # grid lines — gray
    [-1, "#374151"],   # bbox outline — dark
    [10, "#d73027"],
    [20, "#fee08b"],
    [30, "#ffffbf"],
    [40, "#a6d96a"],
    [50, "#1a9850"],
]

bbox_viz = LinestringViz(
    {"type": "FeatureCollection", "features": bbox_features},
    access_token=MAPBOX_TOKEN,
    color_property="_average_speed",
    color_stops=color_stops,
    center=(center_lon, center_lat),
    zoom=zoom,
    line_width_default=2.0,
    opacity=0.9,
)
bbox_viz.show()